# Chinese to English Translation using Pre-trained Model

This notebook uses a pre-trained translation model with GPU acceleration (RTX 3080 Ti) to translate Chinese sentences to English.

## Install Required Packages

Install the necessary packages for GPU-accelerated translation.

In [2]:
%pip install torch torchvision torchaudio #--index-url https://download.pytorch.org/whl/cu121
%pip install transformers pandas accelerate sentencepiece sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 16.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 37.5 MB/s  0:00:22m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 34.1 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 37.9 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 36.9 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 30.6 MB/s  0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.5.1
    Uninstalling triton-3.5.1:
      Successfully uninstalled triton-3.5.1
  Attempting uninstall: nvidia-nvshmem-cu12━━━━━ 0/7 [triton]
    Found existing installation: nvidia-nvshmem-cu12 3.3.200m [triton]
    Uninstalling nvidia-nvshmem-cu12-3.3.20: 0/7 [triton]
      Successfully uninstalled nvidia-nvshmem-cu12-3.3.20 [triton]
  Attempting uninstall: torch╺━━━━━━━━━━━━━━━━━━━━━━ 3/7 [cuda-binding

## Import Libraries and Setup

In [6]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Using device: cuda
GPU: NVIDIA GeForce RTX 3080 Ti Laptop GPU
GPU Memory: 15.61 GB


## Load Translation Model

We'll use Meta's NLLB-200 (No Language Left Behind) model for Chinese to English translation. This is a state-of-the-art multilingual neural machine translation model supporting 200 languages.

In [8]:
# Unload the current model to free GPU memory
del model
del tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
print("Previous model unloaded, GPU memory cleared")

Previous model unloaded, GPU memory cleared


In [9]:
# Load pre-trained NLLB-200 model for Chinese to English translation
# Options: facebook/nllb-200-distilled-600M (faster), facebook/nllb-200-distilled-1.3B (balanced), facebook/nllb-200-3.3B (best quality)
model_name = "facebook/nllb-200-distilled-600M"  # Using 600M model - fits in GPU memory
print(f"Loading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()  # Set to evaluation mode

# NLLB language codes
src_lang = "zho_Hant"  # Chinese Traditional
tgt_lang = "eng_Latn"  # English

print("Model loaded successfully!")
print(f"Source language: {src_lang} -> Target language: {tgt_lang}")
if torch.cuda.is_available():
    print(f"GPU Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

Loading model: facebook/nllb-200-3.3B


Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 28.08it/s]


Model loaded successfully!
Source language: zho_Hant -> Target language: eng_Latn
GPU Memory allocated: 15.38 GB


## Load Data

In [10]:
# Load the sentences CSV file
df = pd.read_csv('sentences.csv', sep=';')
print(f"Loaded {len(df)} rows")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

Loaded 12041 rows

Columns: ['word', 'sentence', 'Translation']

First few rows:


,word,sentence,Translation
0,零,家裡沒有錢，所以零用錢也沒有了。,"There is no money at home, so there is no pock..."
1,零,桌子上面沒有東西，只有零個杯子。,"There is nothing on the table, only zero cups."
2,爺爺,我見到我的爺爺在公園散步。,I saw my grandpa taking a walk in the park.
3,爺爺,我見到我的爺爺很開心。,I saw my grandpa was very happy.
4,奶奶,[FAILED],Grandma


## Define Translation Function

Create a batch translation function to efficiently translate sentences using GPU.

In [11]:
import gc

def translate_batch(texts, batch_size=8, max_length=512):
    """
    Translate a list of Chinese texts to English using GPU acceleration with NLLB-200.
    
    Args:
        texts: List of Chinese texts to translate
        batch_size: Number of sentences to process at once
        max_length: Maximum token length for each sentence
    
    Returns:
        List of translated English texts
    """
    translations = []
    
    # Set source language
    tokenizer.src_lang = src_lang
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = texts[i:i + batch_size]
        
        # Tokenize the batch
        inputs = tokenizer(batch, return_tensors="pt", padding=True, 
                          truncation=True, max_length=max_length).to(device)
        
        # Generate translations with target language code
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
        with torch.no_grad():
            translated = model.generate(
                **inputs, 
                forced_bos_token_id=forced_bos_token_id,
                max_length=max_length,
                use_cache=True  # Enable KV cache for efficiency
            )
        
        # Decode translations
        batch_translations = tokenizer.batch_decode(translated, skip_special_tokens=True)
        translations.extend(batch_translations)
        
        # Aggressive memory cleanup to prevent leaks
        del inputs, translated, batch_translations
        
        # Periodic deep cleanup every 100 batches
        if (i // batch_size) % 100 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
    
    # Final cleanup
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return translations

print("Translation function defined with memory leak prevention!")

Translation function defined with memory leak prevention!


## Prepare Data for Translation

Filter and prepare sentences that need translation (those with "[FAILED]" or missing translations).

In [13]:
# Filter rows where sentence is not "[FAILED]"
df_valid = df[df['sentence'] != '[FAILED]'].copy()
print(f"Valid sentences to translate: {len(df_valid)}")

# Extract sentences for translation
sentences_to_translate = df_valid['sentence'].tolist()

print(f"\nExample sentences to translate:")
for i, sent in enumerate(sentences_to_translate[:3], 1):
    print(f"{i}. {sent}")

Valid sentences to translate: 11495

Example sentences to translate:
1. 家裡沒有錢，所以零用錢也沒有了。
2. 桌子上面沒有東西，只有零個杯子。
3. 我見到我的爺爺在公園散步。


In [14]:
# Translate all sentences with optimal batch size for 600M model
print("Starting translation...")
translations = translate_batch(sentences_to_translate, batch_size=16)

print(f"\nTranslation complete! Translated {len(translations)} sentences")
print(f"\nExample translations:")
for i in range(min(3, len(translations))):
    print(f"\nChinese: {sentences_to_translate[i]}")
    print(f"English: {translations[i]}")

Starting translation...


Translating:   0%|          | 0/2874 [00:00<?, ?it/s]


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## Run Translation

Translate all sentences using GPU acceleration. This will process them in batches for optimal performance.

In [6]:
# Clear GPU cache before starting translation
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print("GPU cache cleared")
    print(f"GPU Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"GPU Memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

GPU cache cleared
GPU Memory allocated: 2.30 GB
GPU Memory reserved: 2.31 GB


In [7]:
# Translate all sentences with batch size optimized for smaller model
print("Starting translation...")
translations = translate_batch(sentences_to_translate, batch_size=8)

print(f"\nTranslation complete! Translated {len(translations)} sentences")
print(f"\nExample translations:")
for i in range(min(3, len(translations))):
    print(f"\nChinese: {sentences_to_translate[i]}")
    print(f"English: {translations[i]}")

Starting translation...


Translating: 100%|██████████| 1437/1437 [05:12<00:00,  4.59it/s]


Translation complete! Translated 11495 sentences

Example translations:

Chinese: 家裡沒有錢，所以零用錢也沒有了。
English: The first time I saw a woman in the house, I was in a coma and I was in a coma.

Chinese: 桌子上面沒有東西，只有零個杯子。
English: The first thing I noticed was that there was nothing on the table, only zero glasses.

Chinese: 我見到我的爺爺在公園散步。
English: I saw my grandfather walking in the park.


## Create Dataset with Schema: idx, word, sentence, translation

Build the final dataset with the specified schema and save it to CSV.

In [38]:
# Add translations to the dataframe
df_valid['translation'] = translations

# Create the final dataset with the specified schema
translated_sentences = pd.DataFrame({
    'idx': range(len(df_valid)),
    'word': df_valid['word'].values,
    'sentence': df_valid['sentence'].values,
    'translation': df_valid['translation'].values
})

print(f"Created dataset with {len(translated_sentences)} rows")
print(f"\nSchema: {list(translated_sentences.columns)}")
print(f"\nFirst few rows:")
translated_sentences.head(10)

Created dataset with 11495 rows

Schema: ['idx', 'word', 'sentence', 'translation']

First few rows:


,idx,word,sentence,translation
0,0,零,家裡沒有錢，所以零用錢也沒有了。,The first thing I did was to get a job in a ba...
1,1,零,桌子上面沒有東西，只有零個杯子。,"The food was very good, and the food was very ..."
2,2,爺爺,我見到我的爺爺在公園散步。,I saw my grandfather walking in the park.
3,3,爺爺,我見到我的爺爺很開心。,I saw my grandfather happy.
4,4,奶奶,我喜歡跟我的奶奶一起喫點心。,I love to eat dessert with my grandmother.
5,5,別人,我不要給別人玩我的玩具。,I don't give my toys to others.
6,6,別人,我不要給別人玩我的玩具。,I don't give my toys to others.
7,7,老人,老人常常坐在公園裡聊天。,The old man often sat in the park and talked.
8,8,年輕,我今年二十歲，還很年輕。,I am 20 years old and still very young.
9,9,男生,這些是男生成羣玩遊戲。,The video game is a male-generated puzzle game.


## Save the Dataset

Save the translated sentences to a CSV file.

In [39]:
# Save to CSV
output_file = 'translated_sentences.csv'
translated_sentences.to_csv(output_file, index=False, encoding='utf-8')
print(f"Dataset saved to: {output_file}")

# Display summary statistics
print(f"\n=== Summary Statistics ===")
print(f"Total sentences translated: {len(translated_sentences)}")
print(f"Unique words: {translated_sentences['word'].nunique()}")
print(f"Average sentence length: {translated_sentences['sentence'].str.len().mean():.1f} characters")
print(f"Average translation length: {translated_sentences['translation'].str.len().mean():.1f} characters")

Dataset saved to: translated_sentences.csv

=== Summary Statistics ===
Total sentences translated: 11495
Unique words: 5608
Average sentence length: 13.9 characters
Average translation length: 51.3 characters
